### 1. add_messages 란

- `messages` 상태 키에 붙이는 **리듀서(reducer)** → 노드가 반환한 메시지를 기존 리스트에 **누적(append)**
- `Annotated[list, add_messages]`로 지정 (주석 없는 키는 매 업데이트마다 덮어써짐)
- 리듀서는 `StateGraph` 실행 중에만 작동 → 아래처럼 dict 직접 대입엔 관여 안 함

In [6]:
from typing import Annotated, TypedDict
from langgraph.graph import add_messages

class MyData(TypedDict):
    # messages 키에 add_messages 리듀서 지정 → 누적(append) 동작
    messages: Annotated[list, add_messages]

# 주의) 리듀서는 그래프 실행 중에만 작동 → dict 직접 대입은 그냥 저장만 됨
data: MyData = {
    "messages": ["메세지01", "메세지02"]
}

print(data)

{'messages': ['메세지01', '메세지02']}


### 2. 메시지 병합 (다른 id → 추가)

- `add_messages(left, right)` → `right`를 `left`에 병합
- id가 서로 다르면 뒤 메시지가 그대로 append

In [7]:
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import add_messages

msg1 = [HumanMessage(content="Hello는 대한민국에서 어떤 말로 대체 가능하니?", id="1")]
msg2 = [AIMessage(content="안녕하세요로 대체 가능합니다", id="2")]

result = add_messages(msg1, msg2)   # id 다름 → [msg1, msg2] 로 병합
print(result)

[HumanMessage(content='Hello는 대한민국에서 어떤 말로 대체 가능하니?', additional_kwargs={}, response_metadata={}, id='1'), AIMessage(content='안녕하세요로 대체 가능합니다', additional_kwargs={}, response_metadata={}, id='2', tool_calls=[], invalid_tool_calls=[])]


### 3. 같은 id → 교체

- `right`에 `left`와 **같은 id**가 있으면 그 메시지를 교체 (없으면 append)
- 그래서 "append-only + id 기준 갱신" 동작

In [3]:
from langchain_core.messages import HumanMessage
from langgraph.graph import add_messages

msg1 = [HumanMessage(content="Hello", id="1")]
msg2 = [HumanMessage(content="Hi", id="1")]    # id=1 → msg1 교체
msg3 = [HumanMessage(content="bye", id="2")]   # id=2 → 추가

msgs = add_messages(msg1, msg2)     # id=1 교체 → [Hi]
result = add_messages(msgs, msg3)   # id=2 추가 → [Hi, bye]
print(result)

[HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='1'), HumanMessage(content='bye', additional_kwargs={}, response_metadata={}, id='2')]
